In [ ]:
import pandas as pd
from neo4j import GraphDatabase

# 1. Connect to your Neo4j database
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "your_password")
driver = GraphDatabase.driver(URI, auth=AUTH)

# 2. Define a batch importer function
def batched_import(statement, df, batch_size=1000):
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start+batch_size]
        driver.execute_query(
            f"UNWIND $rows AS value {statement}", 
            rows=batch.to_dict('records'), 
            database_="neo4j"
        )

# 3. Read the GraphRAG Nodes Parquet file
# Adjust the path to where your GraphRAG output artifacts are stored
df_nodes = pd.read_parquet('output/artifacts/create_final_nodes.parquet')

# 4. Ingest Nodes into Neo4j
node_statement = """
MERGE (n:Entity {id: value.id})
SET n += value {.level, .title, .type, .description}
"""
batched_import(node_statement, df_nodes)